# 使用 PyTorch nn.Module 构建复杂模型

> 本笔记本是 [03-函数式API构建模型.ipynb](./03-函数式API构建模型.ipynb) 的 **PyTorch 等价版本**。
> 原版使用 Keras Functional API，本版使用 PyTorch `nn.Module` 子类模式实现相同功能。

本教程介绍如何使用 PyTorch 的 `nn.Module` 子类化方式构建复杂模型，
包括非线性拓扑结构（Wide & Deep）、多输入和多输出模型。

## 学习目标

1. 理解 `nn.Module` 子类化与 `forward()` 方法的使用
2. 掌握多输入模型的构建方法（`forward()` 接收元组）
3. 学会构建多输出模型（辅助输出，`forward()` 返回元组）
4. 理解 Wide & Deep 架构的设计理念
5. 掌握 PyTorch 手动加权损失的计算方式

## Keras Functional API vs PyTorch nn.Module

| 特性 | Keras Functional API | PyTorch nn.Module |
|------|---------------------|-------------------|
| 模型定义方式 | 声明式（层作为函数连接） | 命令式（子类化 + forward） |
| 多输入 | `inputs=[A, B]` 传给 `Model()` | `forward()` 接收多个参数或元组 |
| 多输出 | `outputs=[main, aux]` 传给 `Model()` | `forward()` 返回元组 |
| 层合并 | `keras.layers.Concatenate()` | `torch.cat()` |
| 损失加权 | `loss_weights=[0.9, 0.1]` | 手动计算 `0.9*main + 0.1*aux` |
| 模型可视化 | `model.summary()` / `keras.utils.plot_model()` | `print(model)` / `torchinfo.summary()` |

## 1. 环境配置与数据准备

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# 设置随机种子 / Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# 设备选择 / Device selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch版本: {torch.__version__}")
print(f"使用设备: {device}")

In [ ]:
# 加载并预处理数据 / Load and preprocess the California Housing dataset
housing = fetch_california_housing()

# 划分数据集 / Split into train/valid/test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=RANDOM_SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_SEED
)

# 标准化 / Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

print(f"训练集: {X_train.shape}")
print(f"验证集: {X_valid.shape}")
print(f"测试集: {X_test.shape}")
print(f"特征数: {X_train.shape[1]}")

In [ ]:
# 转换为 PyTorch 张量 / Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
y_valid_t = torch.tensor(y_valid, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 创建 DataLoader / Create DataLoaders
BATCH_SIZE = 32
train_ds = TensorDataset(X_train_t, y_train_t)
valid_ds = TensorDataset(X_valid_t, y_valid_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"训练批次数: {len(train_loader)}")
print(f"验证批次数: {len(valid_loader)}")
print(f"测试批次数: {len(test_loader)}")

## 2. nn.Module 基础

### 基本语法

PyTorch 通过子类化 `nn.Module` 并实现 `forward()` 方法来定义模型：

```python
# 在 __init__ 中定义层 / Define layers in __init__
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(n_features, 30)
        self.hidden2 = nn.Linear(30, 30)
        self.output = nn.Linear(30, 1)

    # 在 forward 中定义计算流程 / Define computation flow in forward
    def forward(self, x):
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        return self.output(x)
```

### 与 Keras Functional API 的对应关系

| Keras | PyTorch |
|-------|---------|
| `Input(shape=...)` | 在 `__init__` 中定义 `nn.Linear(in_features, ...)` |
| `Dense(30, activation='relu')(x)` | `torch.relu(self.linear(x))` |
| `Concatenate()([a, b])` | `torch.cat([a, b], dim=1)` |
| `Model(inputs=..., outputs=...)` | `nn.Module` 子类 + `forward()` |

In [ ]:
# 使用 nn.Module 构建简单模型 / Build a simple model with nn.Module

class BasicModel(nn.Module):
    """简单的前馈神经网络 / Simple feedforward neural network.

    等价于 Keras Functional API 的基本用法：
    Input -> Dense(30, relu) -> Dense(30, relu) -> Dense(1)
    """
    def __init__(self, n_features: int):
        """初始化模型层 / Initialize model layers.

        Args:
            n_features: 输入特征数 / Number of input features
        """
        super().__init__()
        self.hidden1 = nn.Linear(n_features, 30)
        self.hidden2 = nn.Linear(30, 30)
        self.output = nn.Linear(30, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """前向传播 / Forward pass.

        Args:
            x: 输入张量，形状 (batch, n_features) / Input tensor

        Returns:
            输出张量，形状 (batch, 1) / Output tensor
        """
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        return self.output(x)


n_features = X_train.shape[1]
model_basic = BasicModel(n_features).to(device)

# 模型可视化 / Model visualization
print(model_basic)
print(f"\n总参数量: {sum(p.numel() for p in model_basic.parameters()):,}")
print(f"可训练参数量: {sum(p.numel() for p in model_basic.parameters() if p.requires_grad):,}")

## 3. Wide & Deep 模型

### 架构介绍

Wide & Deep 模型由 Google 在 2016 年提出，结合了：
- **Wide 部分**: 线性模型，擅长记忆（memorization）特征组合
- **Deep 部分**: 深度神经网络，擅长泛化（generalization）

### 架构图

```
         输入特征
        /        \
       /          \
    Wide路径     Deep路径
      |           |
      |        Hidden1
      |           |
      |        Hidden2
       \          /
        \        /
         Concatenate
             |
           输出
```

In [ ]:
# 构建 Wide & Deep 模型 - 单输入版本 / Build Wide & Deep model - single input version

class WideAndDeepModel(nn.Module):
    """Wide & Deep 模型（单输入版本）/ Wide & Deep model (single input version).

    等价于 Keras 代码：
    input -> [input, hidden2] -> concat -> output
                  |
              hidden1 -> hidden2
    """
    def __init__(self, n_features: int):
        """初始化模型层 / Initialize model layers.

        Args:
            n_features: 输入特征数 / Number of input features
        """
        super().__init__()
        # Deep 路径 / Deep path
        self.deep_hidden1 = nn.Linear(n_features, 30)
        self.deep_hidden2 = nn.Linear(30, 30)
        # 合并后的输出层 / Output layer after concatenation
        # Wide 路径直接传递原始输入，与 Deep 路径的 hidden2 输出拼接
        # 拼接后维度 = n_features (wide) + 30 (deep)
        self.output = nn.Linear(n_features + 30, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """前向传播 / Forward pass.

        Args:
            x: 输入张量，形状 (batch, n_features) / Input tensor

        Returns:
            输出张量，形状 (batch, 1) / Output tensor
        """
        # Deep 路径 / Deep path
        deep = torch.relu(self.deep_hidden1(x))
        deep = torch.relu(self.deep_hidden2(deep))
        # Wide 路径直接使用原始输入 / Wide path uses raw input directly
        # 拼接 Wide 和 Deep / Concatenate wide and deep
        concat = torch.cat([x, deep], dim=1)  # dim=1 沿特征维度拼接
        return self.output(concat)


model_wide_deep = WideAndDeepModel(n_features).to(device)
print(model_wide_deep)
print(f"\n总参数量: {sum(p.numel() for p in model_wide_deep.parameters()):,}")

In [ ]:
# 通用训练函数 / Generic training function

def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    n_epochs: int = 30,
    device: torch.device = device,
) -> dict:
    """训练 PyTorch 模型并记录历史 / Train a PyTorch model and record history.

    Args:
        model: 要训练的模型 / Model to train
        train_loader: 训练数据加载器 / Training data loader
        valid_loader: 验证数据加载器 / Validation data loader
        criterion: 损失函数 / Loss function
        optimizer: 优化器 / Optimizer
        n_epochs: 训练轮数 / Number of epochs
        device: 计算设备 / Computing device

    Returns:
        包含训练历史的字典 / Dictionary containing training history
    """
    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}

    for epoch in range(1, n_epochs + 1):
        # 训练阶段 / Training phase
        model.train()
        train_loss_sum = 0.0
        train_mae_sum = 0.0
        train_count = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            y_pred = model(batch_x)
            loss = criterion(y_pred, batch_y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * batch_x.size(0)
            train_mae_sum += torch.abs(y_pred - batch_y).sum().item()
            train_count += batch_x.size(0)

        # 验证阶段 / Validation phase
        model.eval()
        val_loss_sum = 0.0
        val_mae_sum = 0.0
        val_count = 0

        with torch.no_grad():
            for batch_x, batch_y in valid_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)

                y_pred = model(batch_x)
                loss = criterion(y_pred, batch_y)

                val_loss_sum += loss.item() * batch_x.size(0)
                val_mae_sum += torch.abs(y_pred - batch_y).sum().item()
                val_count += batch_x.size(0)

        # 记录指标 / Record metrics
        train_loss = train_loss_sum / train_count
        val_loss = val_loss_sum / val_count
        train_mae = train_mae_sum / train_count
        val_mae = val_mae_sum / val_count

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_mae'].append(train_mae)
        history['val_mae'].append(val_mae)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{n_epochs} | "
                  f"Train Loss: {train_loss:.4f} MAE: {train_mae:.4f} | "
                  f"Val Loss: {val_loss:.4f} MAE: {val_mae:.4f}")

    return history

In [ ]:
# 编译并训练 Wide & Deep 模型 / Compile and train Wide & Deep model
criterion = nn.MSELoss()
optimizer_wd = optim.SGD(model_wide_deep.parameters(), lr=1e-3)

history_wd = train_model(
    model_wide_deep, train_loader, valid_loader,
    criterion, optimizer_wd, n_epochs=30
)

## 4. 多输入模型

在实际应用中，我们可能希望将不同类型的特征分别送入不同的网络路径。

### 示例场景

将 California Housing 数据集的 8 个特征分为两组：
- **Wide 输入 (5 个特征)**: 前 5 个特征直接送入 Wide 路径
- **Deep 输入 (6 个特征)**: 后 6 个特征送入 Deep 路径

注意：某些特征可能同时出现在两个输入中（特征 2-4 重叠）

In [ ]:
# 准备多输入数据 / Prepare multi-input data
# Wide 输入：特征 0-4（共 5 个）
# Deep 输入：特征 2-7（共 6 个）

X_train_A, X_train_B = X_train[:, :5], X_train[:, 2:]
X_valid_A, X_valid_B = X_valid[:, :5], X_valid[:, 2:]
X_test_A, X_test_B = X_test[:, :5], X_test[:, 2:]

print(f"Wide 输入形状: {X_train_A.shape}")
print(f"Deep 输入形状: {X_train_B.shape}")

# 转换为 PyTorch 张量 / Convert to PyTorch tensors
X_train_A_t = torch.tensor(X_train_A, dtype=torch.float32)
X_train_B_t = torch.tensor(X_train_B, dtype=torch.float32)
X_valid_A_t = torch.tensor(X_valid_A, dtype=torch.float32)
X_valid_B_t = torch.tensor(X_valid_B, dtype=torch.float32)
X_test_A_t = torch.tensor(X_test_A, dtype=torch.float32)
X_test_B_t = torch.tensor(X_test_B, dtype=torch.float32)

# 创建多输入 DataLoader / Create multi-input DataLoaders
train_ds_multi = TensorDataset(X_train_A_t, X_train_B_t, y_train_t)
valid_ds_multi = TensorDataset(X_valid_A_t, X_valid_B_t, y_valid_t)
test_ds_multi = TensorDataset(X_test_A_t, X_test_B_t, y_test_t)

train_loader_multi = DataLoader(train_ds_multi, batch_size=BATCH_SIZE, shuffle=True)
valid_loader_multi = DataLoader(valid_ds_multi, batch_size=BATCH_SIZE, shuffle=False)
test_loader_multi = DataLoader(test_ds_multi, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# 构建多输入 Wide & Deep 模型 / Build multi-input Wide & Deep model

class MultiInputWideDeepModel(nn.Module):
    """多输入 Wide & Deep 模型 / Multi-input Wide & Deep model.

    等价于 Keras 代码：
    input_A (wide) ─────────────────┐
                                     ├── Concatenate -> output
    input_B (deep) -> hidden1 -> hidden2 ┘

    Keras 中使用 inputs=[input_A, input_B] 传给 Model()，
    PyTorch 中 forward() 接收两个张量参数。
    """
    def __init__(self, n_wide_features: int = 5, n_deep_features: int = 6):
        """初始化模型层 / Initialize model layers.

        Args:
            n_wide_features: Wide 路径输入特征数 / Number of wide input features
            n_deep_features: Deep 路径输入特征数 / Number of deep input features
        """
        super().__init__()
        # Deep 路径 / Deep path
        self.deep_hidden1 = nn.Linear(n_deep_features, 30)
        self.deep_hidden2 = nn.Linear(30, 30)
        # 合并后的输出层 / Output layer after concatenation
        # 拼接维度 = n_wide_features (wide) + 30 (deep)
        self.output = nn.Linear(n_wide_features + 30, 1)

    def forward(self, x_wide: torch.Tensor, x_deep: torch.Tensor) -> torch.Tensor:
        """前向传播 / Forward pass.

        Args:
            x_wide: Wide 路径输入，形状 (batch, n_wide_features) / Wide path input
            x_deep: Deep 路径输入，形状 (batch, n_deep_features) / Deep path input

        Returns:
            输出张量，形状 (batch, 1) / Output tensor
        """
        # Deep 路径 / Deep path
        deep = torch.relu(self.deep_hidden1(x_deep))
        deep = torch.relu(self.deep_hidden2(deep))
        # 拼接 Wide 和 Deep / Concatenate wide and deep
        concat = torch.cat([x_wide, deep], dim=1)
        return self.output(concat)


model_multi_input = MultiInputWideDeepModel(n_wide_features=5, n_deep_features=6).to(device)
print(model_multi_input)
print(f"\n总参数量: {sum(p.numel() for p in model_multi_input.parameters()):,}")

In [ ]:
# 多输入训练函数 / Multi-input training function

def train_multi_input_model(
    model: nn.Module,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    n_epochs: int = 30,
    device: torch.device = device,
) -> dict:
    """训练多输入 PyTorch 模型 / Train a multi-input PyTorch model.

    DataLoader 中每个批次返回 (x_A, x_B, y)，
    forward() 调用方式为 model(x_A, x_B)。

    Args:
        model: 多输入模型 / Multi-input model
        train_loader: 训练数据加载器（返回 x_A, x_B, y）/ Training data loader
        valid_loader: 验证数据加载器 / Validation data loader
        criterion: 损失函数 / Loss function
        optimizer: 优化器 / Optimizer
        n_epochs: 训练轮数 / Number of epochs
        device: 计算设备 / Computing device

    Returns:
        包含训练历史的字典 / Dictionary containing training history
    """
    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}

    for epoch in range(1, n_epochs + 1):
        # 训练阶段 / Training phase
        model.train()
        train_loss_sum = 0.0
        train_mae_sum = 0.0
        train_count = 0

        for x_A, x_B, batch_y in train_loader:
            x_A, x_B, batch_y = x_A.to(device), x_B.to(device), batch_y.to(device)

            optimizer.zero_grad()
            y_pred = model(x_A, x_B)  # 多输入调用 / Multi-input call
            loss = criterion(y_pred, batch_y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * x_A.size(0)
            train_mae_sum += torch.abs(y_pred - batch_y).sum().item()
            train_count += x_A.size(0)

        # 验证阶段 / Validation phase
        model.eval()
        val_loss_sum = 0.0
        val_mae_sum = 0.0
        val_count = 0

        with torch.no_grad():
            for x_A, x_B, batch_y in valid_loader:
                x_A, x_B, batch_y = x_A.to(device), x_B.to(device), batch_y.to(device)

                y_pred = model(x_A, x_B)
                loss = criterion(y_pred, batch_y)

                val_loss_sum += loss.item() * x_A.size(0)
                val_mae_sum += torch.abs(y_pred - batch_y).sum().item()
                val_count += x_A.size(0)

        train_loss = train_loss_sum / train_count
        val_loss = val_loss_sum / val_count
        train_mae = train_mae_sum / train_count
        val_mae = val_mae_sum / val_count

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_mae'].append(train_mae)
        history['val_mae'].append(val_mae)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{n_epochs} | "
                  f"Train Loss: {train_loss:.4f} MAE: {train_mae:.4f} | "
                  f"Val Loss: {val_loss:.4f} MAE: {val_mae:.4f}")

    return history

In [ ]:
# 训练多输入模型 / Train multi-input model
optimizer_multi = optim.SGD(model_multi_input.parameters(), lr=1e-3)

history_multi = train_multi_input_model(
    model_multi_input, train_loader_multi, valid_loader_multi,
    criterion, optimizer_multi, n_epochs=30
)

In [ ]:
# 评估多输入模型 / Evaluate multi-input model
model_multi_input.eval()
test_loss_sum = 0.0
test_mae_sum = 0.0
test_count = 0

with torch.no_grad():
    for x_A, x_B, batch_y in test_loader_multi:
        x_A, x_B, batch_y = x_A.to(device), x_B.to(device), batch_y.to(device)
        y_pred = model_multi_input(x_A, x_B)
        loss = criterion(y_pred, batch_y)
        test_loss_sum += loss.item() * x_A.size(0)
        test_mae_sum += torch.abs(y_pred - batch_y).sum().item()
        test_count += x_A.size(0)

test_mse = test_loss_sum / test_count
test_mae = test_mae_sum / test_count
print(f"测试集 MSE: {test_mse:.4f}")
print(f"测试集 MAE: {test_mae:.4f}")

# 使用模型预测 / Make predictions
model_multi_input.eval()
with torch.no_grad():
    X_new_A = X_test_A_t[:3].to(device)
    X_new_B = X_test_B_t[:3].to(device)
    y_pred = model_multi_input(X_new_A, X_new_B).cpu().numpy().flatten()

print(f"\n预测值: {y_pred}")
print(f"实际值: {y_test[:3]}")

## 5. 多输出模型（辅助输出）

### 设计理念

在深度网络中添加辅助输出有以下好处：

1. **正则化效果**: 确保底层网络学到有用的特征
2. **梯度流动**: 为网络提供额外的梯度信号
3. **多任务学习**: 同时优化多个相关目标

### 架构设计

```
    Wide输入      Deep输入
        \           |
         \       Hidden1
          \         |
           \     Hidden2 -----> 辅助输出
            \       /
           Concatenate
                |
            主输出
```

### Keras vs PyTorch 损失加权

Keras 中使用 `loss_weights=[0.9, 0.1]` 自动加权，
PyTorch 中需要手动计算：`total_loss = 0.9 * main_loss + 0.1 * aux_loss`

In [ ]:
# 构建多输入多输出模型 / Build multi-input multi-output model

class MultiOutputWideDeepModel(nn.Module):
    """多输入多输出 Wide & Deep 模型 / Multi-input multi-output Wide & Deep model.

    等价于 Keras 代码：
    input_A (wide) ─────────────────┐
                                     ├── Concatenate -> main_output
    input_B (deep) -> hidden1 -> hidden2 ┘
                            |
                        aux_output

    Keras 中使用 outputs=[main_output, aux_output] 传给 Model()，
    PyTorch 中 forward() 返回元组 (main_output, aux_output)。
    """
    def __init__(self, n_wide_features: int = 5, n_deep_features: int = 6):
        """初始化模型层 / Initialize model layers.

        Args:
            n_wide_features: Wide 路径输入特征数 / Number of wide input features
            n_deep_features: Deep 路径输入特征数 / Number of deep input features
        """
        super().__init__()
        # Deep 路径 / Deep path
        self.deep_hidden1 = nn.Linear(n_deep_features, 30)
        self.deep_hidden2 = nn.Linear(30, 30)
        # 主输出：基于合并后的特征 / Main output: based on concatenated features
        self.main_output = nn.Linear(n_wide_features + 30, 1)
        # 辅助输出：直接从 Deep 路径的 hidden2 输出 / Auxiliary output from deep hidden2
        self.aux_output = nn.Linear(30, 1)

    def forward(
        self, x_wide: torch.Tensor, x_deep: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """前向传播 / Forward pass.

        Args:
            x_wide: Wide 路径输入，形状 (batch, n_wide_features) / Wide path input
            x_deep: Deep 路径输入，形状 (batch, n_deep_features) / Deep path input

        Returns:
            (main_output, aux_output) 元组 / Tuple of main and auxiliary outputs
        """
        # Deep 路径 / Deep path
        deep = torch.relu(self.deep_hidden1(x_deep))
        deep = torch.relu(self.deep_hidden2(deep))
        # 拼接 Wide 和 Deep / Concatenate wide and deep
        concat = torch.cat([x_wide, deep], dim=1)
        # 主输出 / Main output
        main_out = self.main_output(concat)
        # 辅助输出 / Auxiliary output
        aux_out = self.aux_output(deep)
        return main_out, aux_out


model_multi_output = MultiOutputWideDeepModel(n_wide_features=5, n_deep_features=6).to(device)
print(model_multi_output)
print(f"\n总参数量: {sum(p.numel() for p in model_multi_output.parameters()):,}")
print("\n各层参数明细 / Layer parameter details:")
for name, param in model_multi_output.named_parameters():
    print(f"  {name}: {param.shape} ({param.numel():,} params)")

In [ ]:
# 多输出训练函数 / Multi-output training function

def train_multi_output_model(
    model: nn.Module,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    loss_weights: list[float] = None,
    n_epochs: int = 30,
    device: torch.device = device,
) -> dict:
    """训练多输出 PyTorch 模型 / Train a multi-output PyTorch model.

    手动实现 Keras 的 loss_weights 功能：
    total_loss = loss_weights[0] * main_loss + loss_weights[1] * aux_loss

    Args:
        model: 多输出模型 / Multi-output model
        train_loader: 训练数据加载器 / Training data loader
        valid_loader: 验证数据加载器 / Validation data loader
        criterion: 损失函数 / Loss function
        optimizer: 优化器 / Optimizer
        loss_weights: 各输出的损失权重 / Loss weights for each output
        n_epochs: 训练轮数 / Number of epochs
        device: 计算设备 / Computing device

    Returns:
        包含训练历史的字典 / Dictionary containing training history
    """
    if loss_weights is None:
        loss_weights = [1.0, 1.0]

    history = {
        'train_loss': [], 'val_loss': [],
        'train_main_loss': [], 'train_aux_loss': [],
        'val_main_loss': [], 'val_aux_loss': [],
        'train_main_mae': [], 'train_aux_mae': [],
        'val_main_mae': [], 'val_aux_mae': [],
    }

    for epoch in range(1, n_epochs + 1):
        # 训练阶段 / Training phase
        model.train()
        metrics = {k: 0.0 for k in [
            'main_loss', 'aux_loss', 'total_loss',
            'main_mae', 'aux_mae', 'count'
        ]}

        for x_A, x_B, batch_y in train_loader:
            x_A, x_B, batch_y = x_A.to(device), x_B.to(device), batch_y.to(device)

            optimizer.zero_grad()
            main_pred, aux_pred = model(x_A, x_B)  # 多输出调用 / Multi-output call

            # 分别计算各输出损失 / Compute loss for each output
            main_loss = criterion(main_pred, batch_y)
            aux_loss = criterion(aux_pred, batch_y)

            # 手动加权总损失 / Manually weighted total loss
            # 等价于 Keras 的 loss_weights=[0.9, 0.1]
            total_loss = loss_weights[0] * main_loss + loss_weights[1] * aux_loss

            total_loss.backward()
            optimizer.step()

            n = x_A.size(0)
            metrics['main_loss'] += main_loss.item() * n
            metrics['aux_loss'] += aux_loss.item() * n
            metrics['total_loss'] += total_loss.item() * n
            metrics['main_mae'] += torch.abs(main_pred - batch_y).sum().item()
            metrics['aux_mae'] += torch.abs(aux_pred - batch_y).sum().item()
            metrics['count'] += n

        # 验证阶段 / Validation phase
        model.eval()
        val_metrics = {k: 0.0 for k in [
            'main_loss', 'aux_loss', 'total_loss',
            'main_mae', 'aux_mae', 'count'
        ]}

        with torch.no_grad():
            for x_A, x_B, batch_y in valid_loader:
                x_A, x_B, batch_y = x_A.to(device), x_B.to(device), batch_y.to(device)

                main_pred, aux_pred = model(x_A, x_B)
                main_loss = criterion(main_pred, batch_y)
                aux_loss = criterion(aux_pred, batch_y)
                total_loss = loss_weights[0] * main_loss + loss_weights[1] * aux_loss

                n = x_A.size(0)
                val_metrics['main_loss'] += main_loss.item() * n
                val_metrics['aux_loss'] += aux_loss.item() * n
                val_metrics['total_loss'] += total_loss.item() * n
                val_metrics['main_mae'] += torch.abs(main_pred - batch_y).sum().item()
                val_metrics['aux_mae'] += torch.abs(aux_pred - batch_y).sum().item()
                val_metrics['count'] += n

        # 记录指标 / Record metrics
        c = metrics['count']
        vc = val_metrics['count']

        history['train_loss'].append(metrics['total_loss'] / c)
        history['val_loss'].append(val_metrics['total_loss'] / vc)
        history['train_main_loss'].append(metrics['main_loss'] / c)
        history['train_aux_loss'].append(metrics['aux_loss'] / c)
        history['val_main_loss'].append(val_metrics['main_loss'] / vc)
        history['val_aux_loss'].append(val_metrics['aux_loss'] / vc)
        history['train_main_mae'].append(metrics['main_mae'] / c)
        history['train_aux_mae'].append(metrics['aux_mae'] / c)
        history['val_main_mae'].append(val_metrics['main_mae'] / vc)
        history['val_aux_mae'].append(val_metrics['aux_mae'] / vc)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{n_epochs} | "
                  f"Train Total: {metrics['total_loss']/c:.4f} "
                  f"(Main: {metrics['main_loss']/c:.4f}, Aux: {metrics['aux_loss']/c:.4f}) | "
                  f"Val Total: {val_metrics['total_loss']/vc:.4f} "
                  f"(Main: {val_metrics['main_loss']/vc:.4f}, Aux: {val_metrics['aux_loss']/vc:.4f})")

    return history

In [ ]:
# 训练多输出模型 / Train multi-output model
# loss_weights=[0.9, 0.1] 等价于 Keras 的 loss_weights 参数
optimizer_mo = optim.SGD(model_multi_output.parameters(), lr=1e-3)

history_multi_output = train_multi_output_model(
    model_multi_output, train_loader_multi, valid_loader_multi,
    criterion, optimizer_mo,
    loss_weights=[0.9, 0.1],  # 主输出权重 0.9，辅助输出权重 0.1
    n_epochs=30
)

In [ ]:
# 评估多输出模型 / Evaluate multi-output model
model_multi_output.eval()
test_main_loss = 0.0
test_aux_loss = 0.0
test_main_mae = 0.0
test_aux_mae = 0.0
test_count = 0

with torch.no_grad():
    for x_A, x_B, batch_y in test_loader_multi:
        x_A, x_B, batch_y = x_A.to(device), x_B.to(device), batch_y.to(device)
        main_pred, aux_pred = model_multi_output(x_A, x_B)

        main_loss = criterion(main_pred, batch_y)
        aux_loss = criterion(aux_pred, batch_y)

        n = x_A.size(0)
        test_main_loss += main_loss.item() * n
        test_aux_loss += aux_loss.item() * n
        test_main_mae += torch.abs(main_pred - batch_y).sum().item()
        test_aux_mae += torch.abs(aux_pred - batch_y).sum().item()
        test_count += n

total_loss = 0.9 * (test_main_loss / test_count) + 0.1 * (test_aux_loss / test_count)

print("评估结果:")
print(f"总损失 (加权): {total_loss:.4f}")
print(f"主输出损失: {test_main_loss / test_count:.4f}")
print(f"辅助输出损失: {test_aux_loss / test_count:.4f}")
print(f"主输出 MAE: {test_main_mae / test_count:.4f}")
print(f"辅助输出 MAE: {test_aux_mae / test_count:.4f}")

In [ ]:
# 多输出预测 / Multi-output predictions
model_multi_output.eval()
with torch.no_grad():
    X_new_A = X_test_A_t[:5].to(device)
    X_new_B = X_test_B_t[:5].to(device)
    y_pred_main, y_pred_aux = model_multi_output(X_new_A, X_new_B)
    y_pred_main = y_pred_main.cpu().numpy().flatten()
    y_pred_aux = y_pred_aux.cpu().numpy().flatten()

print("预测结果对比:")
print("=" * 50)
for i in range(5):
    print(f"样本{i+1}: 实际={y_test[i]:.3f}, "
          f"主预测={y_pred_main[i]:.3f}, "
          f"辅助预测={y_pred_aux[i]:.3f}")

## 6. 可视化训练过程

In [ ]:
# 绘制多输出模型的训练曲线 / Plot training curves for multi-output model
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 主输出和辅助输出损失 / Main and auxiliary output losses
axes[0].plot(history_multi_output['train_main_loss'], label='Train Main')
axes[0].plot(history_multi_output['val_main_loss'], label='Val Main')
axes[0].plot(history_multi_output['train_aux_loss'], label='Train Aux', linestyle='--')
axes[0].plot(history_multi_output['val_aux_loss'], label='Val Aux', linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves (Main & Aux)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 总损失 / Total loss
axes[1].plot(history_multi_output['train_loss'], label='Train Total')
axes[1].plot(history_multi_output['val_loss'], label='Val Total')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Total Loss')
axes[1].set_title('Total Loss (Weighted: 0.9*Main + 0.1*Aux)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. TF vs PyTorch 对照

### 模型定义对照

| 概念 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| 模型类 | `keras.Model` | `nn.Module` |
| 输入定义 | `Input(shape=(8,))` | `__init__` 中 `nn.Linear(8, ...)` |
| 全连接层 | `Dense(30, activation='relu')` | `nn.Linear(in, 30)` + `torch.relu()` |
| 拼接层 | `Concatenate()([a, b])` | `torch.cat([a, b], dim=1)` |
| 多输入 | `Model(inputs=[A, B], ...)` | `forward(self, x_A, x_B)` |
| 多输出 | `Model(..., outputs=[main, aux])` | `return main_out, aux_out` |
| 模型摘要 | `model.summary()` | `print(model)` + 参数计数 |

### 训练流程对照

| 概念 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| 编译 | `model.compile(loss, optimizer, metrics)` | 手动选择 `criterion` + `optimizer` |
| 训练 | `model.fit(X, y, epochs=30)` | 手动循环：`forward` -> `loss` -> `backward` -> `step` |
| 评估 | `model.evaluate(X, y)` | 手动循环 + `torch.no_grad()` |
| 预测 | `model.predict(X)` | `model(X)` (需 `model.eval()` + `torch.no_grad()`) |
| 损失加权 | `loss_weights=[0.9, 0.1]` | `0.9 * main_loss + 0.1 * aux_loss` |
| 数据加载 | NumPy 数组直接传入 | `TensorDataset` + `DataLoader` |

### 核心差异总结

1. **声明式 vs 命令式**: Keras Functional API 是声明式的，先定义计算图再运行；PyTorch 是命令式的，计算图在运行时动态构建
2. **自动 vs 手动**: Keras 封装了训练循环（`fit`），PyTorch 需要手动编写训练循环，但提供了更大的灵活性
3. **损失加权**: Keras 通过 `loss_weights` 参数自动处理，PyTorch 需要手动计算加权损失
4. **设备管理**: Keras 自动管理设备，PyTorch 需要显式 `.to(device)`

## 小结

### nn.Module 子类化的核心特点

1. **灵活性**: `forward()` 方法可以定义任意的计算流程
2. **多输入**: `forward()` 接收多个参数，自然支持多输入
3. **多输出**: `forward()` 返回元组，自然支持多输出
4. **显式控制**: 训练循环完全可控，便于调试和定制

### Wide & Deep 架构的应用场景

- **推荐系统**: 结合用户历史行为（记忆）和用户特征（泛化）
- **CTR 预测**: 同时建模稀疏特征交叉和密集特征
- **结构化数据**: 当需要同时学习简单规则和复杂模式时

### 辅助输出的使用建议

- 辅助输出权重通常设置较小（0.1-0.3）
- 主要用于训练时的正则化，推理时可忽略
- 适用于深度网络，帮助解决梯度消失问题

## 练习

### 练习 1：修改 Wide & Deep 架构

尝试修改 `WideAndDeepModel` 的 Deep 路径：
- 将隐藏层从 2 层增加到 3 层
- 在隐藏层之间添加 `nn.BatchNorm1d` 和 `nn.Dropout(0.2)`
- 比较添加正则化前后验证集上的表现

提示：在 `__init__` 中添加 `self.bn1 = nn.BatchNorm1d(30)` 和 `self.dropout = nn.Dropout(0.2)`，
在 `forward()` 中使用 `deep = self.dropout(torch.relu(self.bn1(self.deep_hidden1(x))))`。

### 练习 2：实现层共享

Keras Functional API 的一个重要特性是层共享（同一层被多次调用）。
在 PyTorch 中，层共享是天然的——只需在 `forward()` 中多次调用同一个层。

请构建一个模型，将 Deep 路径中的 `hidden2` 层调用两次（类似循环），
观察参数量是否变化，并思考为什么 PyTorch 的层共享比 Keras 更直观。

### 练习 3：多任务学习 - 不同目标

当前多输出模型中，两个输出预测相同的目标（房价）。
请修改模型，使辅助输出预测一个不同的目标：
- 主输出：预测房价（回归，MSE 损失）
- 辅助输出：预测房价是否高于中位数（分类，BCE 损失）

提示：
- 需要为辅助输出准备二分类标签：`y_aux = (y > np.median(y)).astype(float)`
- 使用不同的损失函数：`main_criterion = nn.MSELoss()`, `aux_criterion = nn.BCEWithLogitsLoss()`
- 辅助输出层不需要激活函数（`BCEWithLogitsLoss` 内置 sigmoid）